In [1]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

import os

# 1. A Colab Secret, if you set one up (key icon in the left sidebar).
#    Costs nothing when it is not there, and saves you the panel after every restart.
try:
    from google.colab import userdata
    os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
except Exception:
    pass

# 2. Otherwise, the panel.
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()          # follow it; never paste the key into a cell

from pinecone import Pinecone
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

print("connected -", len(pc.list_indexes()), "index(es) in this project")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 9.8 MB/s eta 0:00:00
connected - 0 index(es) in this project


In [2]:
query = "Tell me about Apple's products"

documents = [
    "Apple is a popular fruit known for its sweetness and crisp texture.",
    "Apple is known for its innovative products like the iPhone.",
    "Many people enjoy eating apples as a healthy snack.",
    "Apple Inc. has revolutionized the tech industry with its sleek designs and user-friendly interfaces.",
    "An apple a day keeps the doctor away, as the saying goes."
]

for i, d in enumerate(documents):
    print(f"{i}. {d}")

0. Apple is a popular fruit known for its sweetness and crisp texture.
1. Apple is known for its innovative products like the iPhone.
2. Many people enjoy eating apples as a healthy snack.
3. Apple Inc. has revolutionized the tech industry with its sleek designs and user-friendly interfaces.
4. An apple a day keeps the doctor away, as the saying goes.


In [3]:
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3,
    return_documents=True,        # without this you get ids and scores, no text
)

def show_reranked_results(query, matches):
    print(f"Query: {query}\n")
    for i, m in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. Score: {m.score:.4f}")
        print(f"      {m.document.text}\n")

show_reranked_results(query, reranked.data)       # <- .data, not .matches

Query: Tell me about Apple's products

   1. Score: 0.8391
      Apple is known for its innovative products like the iPhone.

   2. Score: 0.2320
      Apple Inc. has revolutionized the tech industry with its sleek designs and user-friendly interfaces.

   3. Score: 0.1743
      Apple is a popular fruit known for its sweetness and crisp texture.



In [4]:
!pip install -q pandas torch transformers

import os, time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

cloud  = os.getenv('PINECONE_CLOUD',  'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')
spec   = ServerlessSpec(cloud=cloud, region=region)

index_name = 'medical-notes-index'

if pc.has_index(name=index_name):      # makes the cell safe to re-run
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,                     # must match the vectors in the data file
    metric='cosine',
    spec=spec,
)
print("created:", index_name)

created: medical-notes-index


In [5]:
import requests, tempfile

URL = ("https://raw.githubusercontent.com/devtlv/Datasets-GEN-AI-Bootcamp/"
       "refs/heads/main/Week%208/W8D2/data.json")

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")
    response = requests.get(URL)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)
    df = pd.read_json(file_path, orient='records', lines=True)   # keep lines=True

print("Data shape:", df.shape)          # <- the pandas attribute asked for
print("columns   :", list(df.columns))
print("vector len:", len(df['values'][0]))
df.head()

Data shape: (100, 3)
columns   : ['id', 'values', 'metadata']
vector len: 384


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [6]:
index = pc.Index(name=index_name)

index.upsert_from_dataframe(df)        # the DataFrame from the previous cell

def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print("Vector count:", vector_count)
    return vector_count > 0            # <- the condition asked for

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

Vector count: 100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

In [7]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(input_question, padding=True,
                              truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)

    embedding = model_output.last_hidden_state[0].mean(dim=0)   # <- 0
    return embedding

question = "what if my patient has leg pain"
query = get_embedding(question).tolist()

results = index.query(vector=[query], top_k=10, include_metadata=True)
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)
print(len(sorted_matches), "matches")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

10 matches


In [8]:
def show_results(question, matches):
    print(f"Question: '{question}'\n\nResults:")
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'      Score: {match["score"]}')          # <- "score"
        print(f'      Metadata: {match["metadata"]}')    # <- "metadata"
        print('')

show_results(question, sorted_matches)

transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join(
            f"{key}: {value}" for key, value in match['metadata'].items()   # <- "metadata"
        )
    }
    for match in results['matches']
]
print(transformed_documents[0])

Question: 'what if my patient has leg pain'

Results:
   1. ID: P0100
      Score: 0.51816
      Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}

   2. ID: P095
      Score: 0.501063049
      Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   3. ID: P047
      Score: 0.501063049
      Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   4. ID: P007
      Score: 0.459310412
      Metadata: {'surgery': 'knee arthroscopy', 'symptoms': 'pain, swelling', 'treatment': 'physical therapy'}

   5. ID: P028
      Score: 0.445996493
      Metadata: {'condition': 'knee pain', 'referral': 'orthopedics'}

   6. ID: P059
      Score: 0.430181175
      Metadata: {'symptoms': 'joint pain', 'treatment': 'NSAIDs, rest'}

   7. ID: P020
      Score: 0.424475759
      Metadata: {'condition': 'sprained ankle', 'tests': 'X-ray'}

   8. ID: P068
      Score: 0.413904846
      Metadata: {'condition': 'broken arm', 'treatment'

In [9]:
refined_query = "what if my patient had knee surgery"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

def show_reranked_results(question, matches):
    print(f"Question: '{question}'\n\nReranked Results:")
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'      Score: {match.score}')                              # <- .score
        print(f'      Reranking Field: {match.document.reranking_field}') # <- .document.reranking_field
        print('')

show_reranked_results(refined_query, reranked_results.data)               # <- .data

before = [m['id'] for m in sorted_matches][:3]
after  = [m.document.id for m in reranked_results.data]
print("search said :", before)
print("rerank says :", after)

Question: 'what if my patient had knee surgery'

Reranked Results:
   1. ID: P007
      Score: 0.18184364
      Reranking Field: surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical therapy

   2. ID: P028
      Score: 0.0054905633
      Reranking Field: condition: knee pain; referral: orthopedics

   3. ID: P059
      Score: 0.0027900378
      Reranking Field: symptoms: joint pain; treatment: NSAIDs, rest

search said : ['P0100', 'P095', 'P047']
rerank says : ['P007', 'P028', 'P059']


In [ ]:
pc.delete_index(name=index_name)
print("remaining indexes:", [i['name'] for i in pc.list_indexes()])